In [ ]:
!pip install undetected-chromedriver

In [ ]:
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!sh -c 'echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list'
!apt-get -y update
!apt-get install -y google-chrome-stable
!pip install selenium beautifulsoup4

OK
Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:4 https://dl.google.com/linux/chrome/deb stable InRelease
Hit:5 https://cli.github.com/packages stable InRelease
Hit:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Reading package lists... Done
W: Target Packages (main/binary-amd64/Packages) is configured multiple times in /etc/apt/sources.list.d/google-chrome.list:3 and /etc/apt/sources.list.d/google-chrome.list:4
W: Target Packages (main/binary-all/Packages) is configured multiple times in /etc/apt/sources.list.d/google-chrome.list:3 and /etc/apt/sour

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import time
import random
import pandas as pd
from urllib.parse import urljoin
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


# ================================
# Setup Selenium Driver
# ================================
def setup_driver():
    chrome_options = Options()

    # Update this line to match your installed browser
    chrome_options.binary_location = "/usr/bin/google-chrome"

    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120 Safari/537.36"
    )

    driver = webdriver.Chrome(options=chrome_options)
    return driver

# ================================
# Crawl only the target article
# ================================
def crawl_exrx_target(index_url):

    print(f"🕸️ Opening: {index_url}")

    driver = setup_driver()

    try:
        driver.get(index_url)

        # Wait up to 10 seconds for the specific article to appear in the DOM
        # We use a RELATIVE XPath: //main//article[2]
        article = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, "//main//article[2]"))
        )

        html = article.get_attribute("innerHTML")
        soup = BeautifulSoup(html, "html.parser")

        links = []

        html = article.get_attribute("innerHTML")

        soup = BeautifulSoup(html, "html.parser")

        links = []

        for a in soup.find_all("a", href=True):

            href = a["href"]

            if "#" in href or "javascript" in href:
                continue

            full_url = urljoin(index_url, href)

            links.append({
                "category": a.get_text(strip=True),
                "url": full_url
            })

        # remove duplicates
        seen = set()
        unique = []

        for item in links:
            if item["url"] not in seen:
                unique.append(item)
                seen.add(item["url"])

        print(f"✅ Extracted {len(unique)} links from target article")

        return unique

    except Exception as e:

        print(f"❌ Scraping failed: {e}")
        return []

    finally:
        driver.quit()


# ================================
# RUN SCRAPER
# ================================

url = "https://exrx.net/Sports/Warmup"

data = crawl_exrx_target(url)

if data:

    df = pd.DataFrame(data)

    print(df.head(20).to_string(index=False))

    df.to_csv("/content/exrx_links.csv", index=False)

    print("\n💾 Saved CSV to /content/exrx_links.csv")

else:

    print("⚠️ No data extracted.")

🕸️ Opening: https://exrx.net/Sports/Warmup
✅ Extracted 11 links from target article
          category                                                        url
     Jumping Jacks             https://exrx.net/Aerobic/Exercises/JumpingJack
       Arms Circle         https://exrx.net/Stretches/Miscellaneous/ArmCircle
     High Knee Jog             https://exrx.net/Aerobic/Exercises/HighKneeRun
     Walking Lunge      https://exrx.net/Stretches/Miscellaneous/WalkingLunge
      Soldier Kick          https://exrx.net/Stretches/Hamstrings/SoldierKick
     Butt Kick Jog                https://exrx.net/Aerobic/Exercises/ButtKick
Walking Ankle Grab https://exrx.net/Stretches/GluteusMaximus/WalkingAnkleGrab
   Lateral Shuffle                 https://exrx.net/Aerobic/Exercises/Shuffle
         Grapevine               https://exrx.net/Aerobic/Exercises/Grapevine
               Run                     https://exrx.net/Aerobic/Exercises/Run
    Sprint to turn            https://exrx.net/Aerobic/Exe

In [ ]:
len(df)

11

In [ ]:
mask = df["category"].str.contains("Walking Ankle Grab|Sprint to turn",regex=True)
df.drop(df[mask].index,inplace=True)

In [ ]:
df.to_csv("/content/scraped_url.csv",index=False)

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import pandas as pd
import json
import time
import random
from tqdm import tqdm
import gc
import os

# ==========================================
# 1. MEMORY-SAFE HTML FETCHER
# ==========================================
def fetch_html_safe(url):
    html = ""
    driver = None
    try:
        chrome_options = Options()
        chrome_options.add_argument("--headless=new")
        chrome_options.add_argument("--no-sandbox")
        chrome_options.add_argument("--disable-dev-shm-usage")
        chrome_options.add_argument("--disable-gpu")
        chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")

        driver = webdriver.Chrome(options=chrome_options)
        driver.get(url)
        time.sleep(3)

        # Cloudflare Check
        if "Just a moment" in driver.title or "Attention Required" in driver.title:
            time.sleep(8)

        html = driver.page_source
    except Exception as e:
        print(f"\n❌ Error fetching {url}: {e}")
    finally:
        if driver:
            driver.quit()

    return html

# ==========================================
# 2. THE DYNAMIC COLUMN PARSER
# ==========================================
def parse_exercise_data(html, url):
    soup = BeautifulSoup(html, 'html.parser')

    # Initialize basic info
    exercise_data = {
        'source_url': url
    }

    # --- A. Get the Title ---
    title_tag = soup.find('h1', class_='page-title') or soup.find('h1')
    exercise_data['exercise_name'] = title_tag.text.strip() if title_tag else "Unknown"

    # --- B. Locate the target layout (div.row -> col-sm-6) ---
    article = soup.find('article') or soup.find('main')
    if not article:
        return exercise_data

    # Find all columns that hold the exercise data
    columns = article.find_all('div', class_='col-sm-6')

    # Fallback just in case the layout changes
    if not columns:
        columns = [article]

    # --- C. Dynamically Extract ALL Features ---
    for col in columns:
        current_feature_name = "general_info"

        # Iterate through the direct elements inside the column sequentially
        for element in col.children:
            # Skip empty text nodes
            if element.name is None:
                continue

            # 1. Handle Tables (usually the Classification table)
            if element.name == 'table':
                for tr in element.find_all('tr'):
                    tds = tr.find_all('td')
                    if len(tds) == 2:
                        key = tds[0].text.strip().replace(':', '').replace(' ', '_').lower()
                        val = tds[1].text.strip()
                        exercise_data[key] = val
                continue

            # 2. Handle Headers (h2, h3, h4) - This becomes a new feature key
            if element.name in ['h2', 'h3', 'h4']:
                current_feature_name = element.text.strip().replace(' ', '_').lower()
                if current_feature_name not in exercise_data:
                    exercise_data[current_feature_name] = []
                continue

            # 3. Handle Paragraphs (Might be a sub-header or just text)
            if element.name == 'p':
                strong_tag = element.find('strong')
                text = element.text.strip()

                if not text:
                    continue

                # If the paragraph is entirely (or mostly) bold, treat it as a new feature key
                if strong_tag and (len(text) - len(strong_tag.text.strip()) < 5):
                    current_feature_name = strong_tag.text.strip().replace(':', '').replace(' ', '_').lower()
                    if current_feature_name not in exercise_data:
                        exercise_data[current_feature_name] = []
                else:
                    # Otherwise, it's just regular text belonging to the current feature
                    if current_feature_name not in exercise_data:
                        exercise_data[current_feature_name] = []
                    exercise_data[current_feature_name].append(text)
                continue

            # 4. Handle Lists (ul, ol) - Append bullet points to the current feature
            if element.name in ['ul', 'ol']:
                if current_feature_name not in exercise_data:
                    exercise_data[current_feature_name] = []
                for li in element.find_all('li'):
                    exercise_data[current_feature_name].append(f"• {li.text.strip()}")

    # --- D. Clean Up Extracted Lists into Strings ---
    for key, val in exercise_data.items():
        if isinstance(val, list):
            # Remove any empty items and join with line breaks
            clean_list = [v for v in val if v]
            if clean_list:
                exercise_data[key] = "\n".join(clean_list)
            else:
                exercise_data[key] = None

    return exercise_data

# ==========================================
# 3. THE MASTER SCRAPE LOOP
# ==========================================
try:
    # Ensure this CSV has at least a 'url' column
    df_master = pd.read_csv('/content/scraped_url.csv')
except FileNotFoundError:
    print("Could not find master list! Ensure 'scraped_url.csv' is in your folder.")
    exit()

output_file = '/content/final_exrx_database.json'
scraped_data_list = []

# If we crashed previously, load the existing data to resume!
if os.path.exists(output_file):
    with open(output_file, 'r', encoding='utf-8') as f:
        scraped_data_list = json.load(f)
    print(f"Resuming from checkpoint! {len(scraped_data_list)} exercises already scraped.")

# Get a list of URLs we have already scraped to avoid double-work
completed_urls = [item.get('source_url') for item in scraped_data_list]

print(f"\n🚀 Beginning Dynamic Content Scrape for {len(df_master)} exercises...\n")

for index, row in tqdm(df_master.iterrows(), total=len(df_master), desc="Scraping Exercises"):
    url = row['url']

    if url in completed_urls:
        continue

    html = fetch_html_safe(url)

    if html:
        # Note: We now only pass html and url. The parser does the rest!
        data = parse_exercise_data(html, url)
        scraped_data_list.append(data)

    # Random human delay
    time.sleep(random.uniform(2.0, 4.0))
    gc.collect()

    # Checkpoint System
    if (index + 1) % 25 == 0:
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(scraped_data_list, f, indent=4, ensure_ascii=False)
        print(f"\n   💾 Checkpoint Saved: {len(scraped_data_list)} exercises secured.")

# ==========================================
# 4. FINAL SAVE
# ==========================================
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(scraped_data_list, f, indent=4, ensure_ascii=False)

print("\n" + "="*50)
print(" 🎉 SCRAPING COMPLETE! 🎉")
print("="*50)
print(f" Total Exercises Extracted: {len(scraped_data_list)}")
print(f" Database Saved To: {output_file}")


🚀 Beginning Dynamic Content Scrape for 9 exercises...



Scraping Exercises: 100%|██████████| 9/9 [01:49<00:00, 12.14s/it]


 🎉 SCRAPING COMPLETE! 🎉
 Total Exercises Extracted: 9
 Database Saved To: /content/final_exrx_database.json


In [ ]:
# !rm /content/final_exrx_database.json
# print("Old database deleted! Ready for a fresh start.")

Old database deleted! Ready for a fresh start.


In [ ]:
dff=pd.read_json("/content/final_exrx_database.json")
dff.columns

Index(['source_url', 'exercise_name', 'classification', 'bearing', 'impact',
       'instructions', 'preparation', 'execution', 'comments',
       'force_(articulation)', 'dynamic', 'utility', 'mechanics', 'force',
       'easier', 'harder', 'static', 'muscles'],
      dtype='object')

In [ ]:
dff.head()

,source_url,exercise_name,classification,bearing,impact,instructions,preparation,execution,comments,force_(articulation),dynamic,utility,mechanics,force,easier,harder,static,muscles
0,https://exrx.net/Aerobic/Exercises/JumpingJack,Jumping Jack,NaN,Weight,High,NaN,"Stand with feet together, knees slightly bent,...",Jump while raising arms and separating legs to...,Intensity can be increased by jumping faster.,NaN,• Ankle\n\t\nPlantar Flexion\n• Plantar Flexio...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,https://exrx.net/Stretches/Miscellaneous/ArmCi...,Arm Circle,NaN,NaN,NaN,NaN,Stand with feet shoulder width or slightly wid...,Rotate arms together in small circular motions...,Circles can rotate either direction (1) down a...,NaN,• Shoulder\n\t\nAdduction\nAbduction\nTransver...,Dynamic Stretch,Isolated,Pull & Push,NaN,NaN,NaN,NaN
2,https://exrx.net/Aerobic/Exercises/HighKneeRun,High Knee Run,NaN,Weight,High,NaN,Stand with left front slightly in front of oth...,Lift right knee high and step forward while sw...,This movement can serve as Warm-up and Dynamic...,NaN,• Hip\n\t\nExtension\nFlexion\n• Extension\n• ...,NaN,NaN,NaN,Pump more slowly without lifting legs as high ...,"Pump more quickly and lift knees high, bringin...",• Hip\n\t\nAbduction\n• Abduction\n• Spine\n\t...,NaN
3,https://exrx.net/Stretches/Miscellaneous/Walki...,Walking Lunge,NaN,NaN,NaN,NaN,Stand facing intended path with arms to sides.,Step forward with right leg and bend left arm....,This movement can serve as Warm-up and Dynamic...,NaN,• Hip\n\t\nExtension\nFlexion\n• Extension\n• ...,Dynamic Stretch,Compound,Push,NaN,NaN,"• Spine (Thoracic, Lumbar)\n\t\nExtension\nLat...",NaN
4,https://exrx.net/Stretches/Hamstrings/SoldierKick,Soldier Kick,NaN,NaN,NaN,NaN,Stand and step forward with left foot.,"Next step, extend left arm forward while swing...",This Dynamic Stretch can performed before at...,NaN,NaN,Dynamic Stretch,Isolated,Pull,NaN,NaN,NaN,Target\n• Hamstrings\nOther\n• Iliopsoas


In [ ]:
import json
from IPython.display import display, HTML

# --- 1. SET THE EXERCISE YOU WANT TO FIND ---
SEARCH_NAME = "Jumping Jack"

# --- 2. LOAD DATABASE ---
file_path = '/content/final_exrx_database.json'
try:
    with open(file_path, 'r', encoding='utf-8') as f:
        database = json.load(f)
except FileNotFoundError:
    print("❌ Could not find the JSON file. Check the path!")
    database = []

# --- 3. DYNAMIC SEARCH & DISPLAY LOGIC ---
if database:
    # Find the first exercise that matches the search name (case-insensitive)
    target_exercise = next((ex for ex in database if SEARCH_NAME.lower() in ex.get('exercise_name', '').lower()), None)

    if target_exercise:
        print(f"✅ Found match for '{SEARCH_NAME}'!\n")

        # Start building the HTML Card
        html_content = f"""
        <div style="font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; max-width: 800px; margin: 20px auto; border: 1px solid #e0e0e0; border-radius: 12px; box-shadow: 0 4px 15px rgba(0,0,0,0.05); background-color: #ffffff; overflow: hidden;">

            <div style="background-color: #2c3e50; color: white; padding: 20px 25px;">
                <h1 style="margin: 0; font-size: 26px; letter-spacing: 0.5px;">{target_exercise.get('exercise_name', 'Unknown Exercise')}</h1>
                <a href="{target_exercise.get('source_url', '#')}" target="_blank" style="color: #63b8ff; text-decoration: none; font-size: 14px; margin-top: 5px; display: inline-block;">🔗 View Original Source on ExRx</a>
            </div>

            <div style="padding: 25px;">
        """

        # --- A. Classification Badges ---
        # Look for standard meta keys and display them as neat pill badges
        meta_keys = ['utility', 'mechanics', 'force']
        meta_html = '<div style="display: flex; gap: 10px; margin-bottom: 25px; flex-wrap: wrap;">'
        has_meta = False

        for key in meta_keys:
            if key in target_exercise and target_exercise[key]:
                has_meta = True
                meta_html += f'<span style="background-color: #e8f4f8; color: #2980b9; padding: 6px 14px; border-radius: 20px; font-size: 13px; font-weight: bold; border: 1px solid #bae1f2;">{key.title()}: {target_exercise[key]}</span>'
        meta_html += '</div>'

        if has_meta:
            html_content += meta_html

        # --- B. Dynamic Content Blocks ---
        # Loop through all other keys extracted by the scraper
        skip_keys = ['exercise_name', 'source_url'] + meta_keys

        for key, value in target_exercise.items():
            if key in skip_keys or not value:
                continue

            # Format the key beautifully (e.g., 'target_muscles' -> 'Target Muscles')
            display_key = key.replace('_', ' ').title()

            # Give muscle-related sections a distinct red color theme, and instruction/info sections a dark theme
            is_muscle = any(m in key for m in ['target', 'synergist', 'stabilizer', 'antagonist'])
            header_color = "#e74c3c" if is_muscle else "#2c3e50"
            bg_color = "#fdf2f0" if is_muscle else "#f8f9fa"
            border_color = "#fadbd8" if is_muscle else "#e9ecef"

            html_content += f"""
            <div style="background-color: {bg_color}; padding: 18px; border-radius: 8px; margin-bottom: 15px; border: 1px solid {border_color}; border-left: 5px solid {header_color};">
                <h3 style="margin-top: 0; margin-bottom: 12px; color: {header_color}; font-size: 18px;">{display_key}</h3>
                <div style="color: #444; line-height: 1.6; white-space: pre-wrap; font-size: 15px;">{value}</div>
            </div>
            """

        # Close out the HTML
        html_content += """
            </div>
        </div>
        """
        display(HTML(html_content))

    else:
        print(f"⚠️ Could not find any exercise matching '{SEARCH_NAME}'.")
        print("Try a different search term!")

✅ Found match for 'Jumping Jack'!



In [ ]:
dff.to_csv("/content/warmup_gym.csv")



In [ ]:
import shutil
import os

file_path=("/content/warmup_gym.csv")

dist_path=("/content/drive/MyDrive/SPORT_METADATA/Structure data/warm_up/Gym")

os.makedirs(dist_path,exist_ok=True)

shutil.copy2(file_path,dist_path)

'/content/drive/MyDrive/SPORT_METADATA/Structure data/warm_up/Gym/warmup_gym.csv'